#  Preprocessing, Modeling, dan Evaluasi

Catatan: Notebook ini merupakan lanjutan dari bagian EDA (pada notebook 01_eda_sepsis.ipynb) yang menghasilkan df_balanced.cs.

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
)

In [ ]:
df_balanced = pd.read_csv("../data/df_balanced.csv")
print(f"Loaded df_balanced: {df_balanced.shape}")
print("Distribusi kelas:")
print(df_balanced["SepsisLabel"].value_counts().to_string())

## 3. Preprocessing - Feature Selection dan Scaling

Tahap ini mempersiapkan df_balanced menjadi matriks numerik yang siap dilatih MLP dalam tiga sub-tahap:
1. Visualisasi korelasi penuh dan deteksi administrative leakage (ICULOS, HospAdmTime, dll).
2. Feature selection: drop kolom administratif dan prune fitur dengan |r| > 0.95.
3. Stratified split dengan komposisi 80/20 dan StandardScaler yang hanya di-fit pada training set.

### 3.1 Full Correlation Heatmap dan Leakage Detection

In [ ]:
corr_full = df_balanced.corr(method="pearson", numeric_only=True)

fig, ax = plt.subplots(figsize=(20, 18))
sns.heatmap(
    corr_full,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    annot=True,
    fmt=".2f",
    annot_kws={"size": 6},
    square=True,
    cbar_kws={"shrink": 0.6},
    linewidths=0.3,
    linecolor="white",
    ax=ax,
)
ax.set_title(f"Full Feature Correlation (df_balanced, n={len(df_balanced):,})", fontsize=14, pad=12)
plt.xticks(rotation=75, ha="right", fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()

heatmap_path = "../reports/full_feature_heatmap.png"
plt.savefig(heatmap_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Berhasil disimpan pada {heatmap_path}")

label_corr = corr_full["SepsisLabel"].drop("SepsisLabel").abs().sort_values(ascending=False)
print("\nTop-10 |Pearson r| vs SepsisLabel (df_balanced):")
print(label_corr.head(10).round(3).to_string())

Heatmap di atas menunjukkan multikolinearitas klinis yang diharapkan, misalnya Hct dan Hgb yang keduanya mengukur konsentrasi sel darah merah. Kolom administratif seperti ICULOS dan HospAdmTime muncul di Top-10 korelasi terhadap SepsisLabel dan akan dibuang di subsection berikutnya.

### 3.2 Feature Selection: Drop Kolom Administratif dan Fitur Redundan

In [ ]:
ADMIN_COLS = ["Patient_ID", "ICULOS", "Hour", "HospAdmTime"]
df_feat = df_balanced.drop(columns=[c for c in ADMIN_COLS if c in df_balanced.columns]).copy()
df_feat = df_feat.loc[:, ~df_feat.columns.str.startswith("Unnamed")]
print(f"Setelah drop admin: {df_feat.shape[1]} kolom ({df_feat.shape[0]:,} baris)")

df_raw_ref = pd.read_csv("../data/Dataset.csv", index_col=0)
raw_missing_pct = (df_raw_ref.isna().mean() * 100).to_dict()
del df_raw_ref

REDUNDANCY_THRESHOLD = 0.95
feat_only = df_feat.drop(columns=["SepsisLabel"])
corr_feat = feat_only.corr().abs()
cols = sorted(corr_feat.columns.tolist())
corr_feat = corr_feat.loc[cols, cols]

redundant_drops = []
dropped_set = set()
print(f"\nScanning pairs dengan |r| > {REDUNDANCY_THRESHOLD} ...")
for i, a in enumerate(cols):
    if a in dropped_set:
        continue
    for b in cols[i + 1:]:
        if b in dropped_set:
            continue
        r = corr_feat.loc[a, b]
        if r > REDUNDANCY_THRESHOLD:
            miss_a = raw_missing_pct.get(a, float("nan"))
            miss_b = raw_missing_pct.get(b, float("nan"))
            if miss_a > miss_b:
                drop, keep = a, b
            elif miss_b > miss_a:
                drop, keep = b, a
            else:
                drop, keep = sorted([a, b])[1], sorted([a, b])[0]
            redundant_drops.append(drop)
            dropped_set.add(drop)
            print(
                f"  Pair |r|={r:.3f}: {a} (miss={miss_a:.2f}%) vs {b} (miss={miss_b:.2f}%) "
                f"  DROP {drop}, KEEP {keep}"
            )
            if a == drop:
                break

if not redundant_drops:
    print("  (tidak ada pasangan yang melampaui threshold)")

df_feat = df_feat.drop(columns=redundant_drops)

assert "SepsisLabel" in df_feat.columns
for c in ADMIN_COLS:
    assert c not in df_feat.columns, f"{c} seharusnya sudah di-drop"

final_features = sorted(df_feat.drop(columns=["SepsisLabel"]).columns.tolist())
print(f"\nJumlah fitur final (tidak termasuk target): {len(final_features)}")
print("Fitur:")
for c in final_features:
    print(f"  - {c}")

### 3.3 Stratified Split dan StandardScaler

In [ ]:
feature_names = df_feat.drop(columns=["SepsisLabel"]).columns.tolist()
X = df_feat[feature_names].values.astype(np.float64)
y = df_feat["SepsisLabel"].astype(int).values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

assert np.isfinite(X_train).all(), "X_train masih mengandung nilai non-finite"
assert np.isfinite(X_test).all(), "X_test masih mengandung nilai non-finite"

train_ratio = X_train.shape[0] / (X_train.shape[0] + X_test.shape[0])
print(f"X_train shape: {X_train.shape}")
print(f"X_test  shape: {X_test.shape}")
print(f"y_train: {y_train.shape} | pos={int(y_train.sum())} ({y_train.mean()*100:.2f}%)")
print(f"y_test:  {y_test.shape} | pos={int(y_test.sum())} ({y_test.mean()*100:.2f}%)")
print(f"Train fraction: {train_ratio:.4f}")
print(f"X_train mean (abs max): {np.abs(X_train.mean(axis=0)).max():.2e}")
print(f"X_train std (abs(1-std) max): {np.abs(X_train.std(axis=0) - 1).max():.2e}")

np.save("../data/X_train.npy", X_train)
np.save("../data/X_test.npy", X_test)
np.save("../data/y_train.npy", y_train)
np.save("../data/y_test.npy", y_test)
np.save("../data/feature_names.npy", np.array(feature_names))
print("\Berhasil disimpan pada data/{X_train, X_test, y_train, y_test, feature_names}.npy")

## 4. Pelatihan Multi-Layer Perceptron

Arsitektur 3 hidden layer (64 neuron, 32 neuron, 16 neuron) dengan fungsi aktivasi ReLU dan optimizer Adam dipilih agar model dapat dengan cukup ekspresif  menangkap interaksi non-linear antar penanda metabolik (Lactate, O2Sat, BUN, Creatinine) tanpa mengalami overfitting terhadap dataset balanced 1:2. Early stopping dengan 10% validation hold-out dan n_iter_no_change=10 digunakan sebagai regularisasi implisit.

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32, 16),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    batch_size=256,
    learning_rate_init=1e-3,
    max_iter=200,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    random_state=42,
    verbose=True,
)
mlp.fit(X_train, y_train)

MODEL_PATH = "../data/sepsis_mlp_model.pkl"
joblib.dump(mlp, MODEL_PATH)
print(f"\nKonvergen pada iterasi ke-{mlp.n_iter_} | best val score: {mlp.best_validation_score_:.4f}")
print(f"Model berhasil disimpan pada {MODEL_PATH}")

## 5. Evaluasi Performa

Evaluasi dilakukan pada test set 20% yang belum pernah dilihat model. Empat plot diagnostik disimpan ke direktori reports/ untuk keperluan laporan: 
1. Loss curve dengan validation accuracy
2. Confusion matrix 
3. ROC curve (AUC) 
4. Precision-Recall curve (Average Precision)

In [ ]:
y_pred  = mlp.predict(X_test)
y_proba = mlp.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["No Sepsis", "Sepsis"], digits=4))
test_f1   = f1_score(y_test, y_pred)
test_prec = precision_score(y_test, y_pred)
test_rec  = recall_score(y_test, y_pred)
test_auc  = roc_auc_score(y_test, y_proba)
test_ap   = average_precision_score(y_test, y_proba)
print(f"F1={test_f1:.4f}  Precision={test_prec:.4f}  Recall={test_rec:.4f}  ROC-AUC={test_auc:.4f}  AP={test_ap:.4f}")

### Loss Curve

In [ ]:
fig, ax1 = plt.subplots(figsize=(7, 4.5))
ax1.plot(mlp.loss_curve_, color="tab:blue", label="Training loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss", color="tab:blue")
if hasattr(mlp, "validation_scores_") and mlp.validation_scores_:
    ax2 = ax1.twinx()
    ax2.plot(mlp.validation_scores_, color="tab:orange", label="Validation accuracy")
    ax2.set_ylabel("Validation accuracy", color="tab:orange")
plt.title("MLP Training - Loss dan Validation Accuracy")
plt.tight_layout()
plt.savefig("../reports/loss_curve.png", dpi=200)
plt.show()

### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4.5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["No Sepsis", "Sepsis"],
    yticklabels=["No Sepsis", "Sepsis"],
    ax=ax,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix - Test Set")
plt.tight_layout()
plt.savefig("../reports/confusion_matrix.png", dpi=200)
plt.show()

### ROC Curve (AUC)

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"MLP (AUC = {test_auc:.4f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.5)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Sepsis MLP")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("../reports/roc_curve.png", dpi=200)
plt.show()

### Precision-Recall curve (Average Precision)

In [ ]:
prec_arr, rec_arr, _ = precision_recall_curve(y_test, y_proba)
plt.figure(figsize=(6, 5))
plt.plot(rec_arr, prec_arr, label=f"MLP (AP = {test_ap:.4f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve - Sepsis MLP")
plt.legend(loc="lower left")
plt.tight_layout()
plt.savefig("../reports/pr_curve.png", dpi=200)
plt.show()

## 6. Explainability dengan SHAP

Setelah MLP terlatih dan tervalidasi, analisis SHAP (KernelExplainer) digunakan untuk menjelaskan bagaimana model memprediksi sepsis. Pertanyaan utama yang ditanyakan dalam proses ini adalah apakah fitur biologis yang mendominasi (Lactate, O2Sat, BUN, Hgb, dll.) atau ada shortcut non-biologis yang terpakai?

Konfigurasi yang digunakan:
- SHAP_SAMPLE_SIZE = 500 baris evaluasi dari X_test
- BACKGROUND_SIZE = 50 baris baseline dari X_train
- NSAMPLES = 100 permutasi per baris evaluasi

In [ ]:
SHAP_SAMPLE_SIZE = 500
BACKGROUND_SIZE  = 50
NSAMPLES         = 100
RANDOM_STATE     = 42

rng = np.random.default_rng(RANDOM_STATE)
bg_idx = rng.choice(X_train.shape[0], size=BACKGROUND_SIZE, replace=False)
ev_idx = rng.choice(X_test.shape[0], size=SHAP_SAMPLE_SIZE, replace=False)

background  = X_train[bg_idx]
eval_sample = X_test[ev_idx]

print(f"X_train: {X_train.shape} | X_test: {X_test.shape} | features: {len(feature_names)}")
print(f"background : {background.shape}")
print(f"eval_sample: {eval_sample.shape}")

In [ ]:
t0 = time.time()
explainer = shap.KernelExplainer(mlp.predict_proba, background)
shap_values = explainer.shap_values(eval_sample, nsamples=NSAMPLES)
elapsed = time.time() - t0

if isinstance(shap_values, list):
    shap_pos = shap_values[1]
else:
    sv_arr = np.asarray(shap_values)
    shap_pos = sv_arr[:, :, 1] if sv_arr.ndim == 3 else sv_arr

print(f"SHAP selesai dalam {elapsed:.1f}s")
print(f"Matriks SHAP positif (Sepsis=1): {shap_pos.shape}")

In [ ]:
plt.figure()
shap.summary_plot(shap_pos, eval_sample, feature_names=feature_names, show=False)
plt.title("Dampak Fitur Klinis terhadap Krisis Metabolik Sepsis (SHAP)")
plt.tight_layout()
plt.savefig("../reports/shap_summary.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: reports/shap_summary.png")

In [ ]:
mean_abs = np.abs(shap_pos).mean(axis=0)
order = np.argsort(mean_abs)[::-1]

print("Top 10 fitur paling berpengaruh (mean |SHAP|):")
print(f"{'Rank':>4}  {'Fitur':<20s}  {'mean|SHAP|':>12s}")
for rank, j in enumerate(order[:10], 1):
    print(f"{rank:>4}  {feature_names[j]:<20s}  {mean_abs[j]:>12.6f}")

### Temuan SHAP

1. Lactate dan O2Sat tidak muncul di Top 10 mean SHAP pada sampel evaluasi 500 baris ini. Model lebih banyak menyandarkan keputusannya pada Temp, BUN, Hgb, pH, Platelets, dan BaseExcess. Hal ini tetap konsisten dengan narasi krisis metabolik dan hipoksia jaringan: BUN, Hgb, pH, dan BaseExcess merupakan penanda hipoperfusi dan asidosis metabolik yang menyertai sepsis berat.
2. Sparsity tinggi pada Lactate (sekitar 97% missing sebelum imputasi) menyebabkan banyak nilai berasal dari median global setelah ffill sehingga sinyal diskriminatifnya tertekan dibanding fitur yang lebih sering diukur seperti Temp, HR, dan Resp.